<a href="https://colab.research.google.com/github/moist234/ECON3916-Statistical-Machine-Learning/blob/main/lab-13/Lab13_HedonicPricing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

In [29]:
# Step 1: Ingestion and Naive Model
url = 'https://raw.githubusercontent.com/moist234/ECON3916-Statistical-Machine-Learning/refs/heads/main/Data/Zillow_California_2026_Hedonic.csv'
df = pd.read_csv(url)

naive_model = smf.ols("Sale_Price ~ Property_Age",data=df).fit()
print(naive_model.summary())
print("\nNaive Age Coefficient:", naive_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     3105.
Date:                Wed, 18 Mar 2026   Prob (F-statistic):          1.26e-308
Time:                        20:12:51   Log-Likelihood:                -12818.
No. Observations:                1000   AIC:                         2.564e+04
Df Residuals:                     998   BIC:                         2.565e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     3.013e+05   7218.570     41.742   

In [30]:
# Step 2: The Multivariate Model
multi_model = smf.ols('Sale_Price ~ Property_Age + Distance_to_Tech_Hub', data=df).fit()
print(multi_model.summary())
print("\nMultivariate Age Coefficient:", multi_model.params['Property_Age'])

# Step 3: FWL Theorem Manual Proof
# 3a: Partial out distance from Price
res_y_model = smf.ols('Sale_Price ~ Distance_to_Tech_Hub', data=df).fit()
df['Price_Residuals'] = res_y_model.resid

# 3b: Partial out distance from Age
res_x_model = smf.ols('Property_Age ~ Distance_to_Tech_Hub', data=df).fit()
df['Age_Residuals'] = res_x_model.resid

# 3c: Regress Residuals on Residuals (-1 removes the intercept for exact mathematical matching)
fwl_model = smf.ols('Price_Residuals ~ Age_Residuals', data=df).fit()
print("\nFWL Isolated Age Coefficient:", fwl_model.params['Age_Residuals'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.954
Method:                 Least Squares   F-statistic:                 1.040e+04
Date:                Wed, 18 Mar 2026   Prob (F-statistic):               0.00
Time:                        20:12:51   Log-Likelihood:                -11982.
No. Observations:                1000   AIC:                         2.397e+04
Df Residuals:                     997   BIC:                         2.399e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept             1.203e+06 

In [31]:
import statsmodels.api as sm
import numpy as np

# Fit the model
X = sm.add_constant(df[['Property_Age', 'Distance_to_Tech_Hub']])
model = sm.OLS(df['Sale_Price'], X).fit()

# Extract coefficients from model.params (indexed by variable name)
b0 = model.params['const']
b1 = model.params['Property_Age']
b2 = model.params['Distance_to_Tech_Hub']

# Build meshgrid — identical logic to the JS above
age_grid, dist_grid = np.meshgrid(
    np.linspace(df.Property_Age.min(), df.Property_Age.max(), 40),   # columns
    np.linspace(df.Distance_to_Tech_Hub.min(), df.Distance_to_Tech_Hub.max(), 40)  # rows
)

# Evaluate regression equation at every grid point → surface Z values
Z = b0 + b1 * age_grid + b2 * dist_grid